In [1]:
MODELS = [
    "esm3_sm_open_v1",
    "esm3-medium-2024-08",
    "esm3-large-2024-03",
    "esmc-6b-2024-12",
    "facebook/esm2_t6_8M_UR50D",
    "facebook/esm2_t12_35M_UR50D",
    "facebook/esm2_t30_150M_UR50D",
    "facebook/esm2_t33_650M_UR50D",
    "facebook/esm2_t36_3B_UR50D",
    "facebook/esm2_t48_15B_UR50D",
    "mint",
]

In [2]:
import lmdb
import json
import pandas as pd
import torch
import dgeb
from pathlib import Path


dfs_lmdb = {}
for data_type in ['train', 'valid', 'test']:
    env = lmdb.open(f'HumanPPI/normal/{data_type}/')
    res = []

    with env.begin() as txn:
        for key, value in txn.cursor():
            if key == b'info' or key == b'length':
                continue
            res.append(json.loads(value))
    
    env.close()

    dfs_lmdb[data_type] = pd.DataFrame(res)

df_lmdb = pd.concat(dfs_lmdb.values())

In [3]:
df_lmdb

,name_1,name_2,seq_1,seq_2,label
0,Q01780,Q9Y333,MAPPSTREPRVLSATSATKSDGEMVLPGFPDADSFVKFALGSVVAV...,MLFYSFFKSLVGKDVVVELKNDLSICGTLHSVDQYLNIKLTDISVT...,1
1,Q9P104,P06213,MASNFNDIVKQGYVRIRSRRLGIYQRCWLVFKKASSKGPKRLEKFS...,MATGGRRGAAAAPLLVAVAALLLGAAGHLYPGEVCPGMDIRNNLTR...,1
2,O00300,P04004,MNNLLCCALVFLDISIKWTTQETFPPKYLHYDEETSHQLLCDKCPP...,MAPLRPLLILALLAWVALADQESCKGRCTEGFNVDKKCQCDELCSY...,1
3,Q9UNY4,P22626,MEEVRCPEHGTFCFLKTGVRDGPNKGKSFYVCRADTCSFVRATDIP...,MEKTLETVPLERKKREKEQFRKLFIGGLSFETTEESLRNYYEQWGK...,1
4,Q15139,Q02156,MSAPPVLRPPSPLLPVAAAAAAAAAALVPGSGPGPAPFLAPVAAPV...,MVVFNGLLKIKICEAVSLKPTAWSLRHAVGPRPQTFLLDPYIALNV...,1
...,...,...,...,...,...
175,P09429,Q92552,MGKGDPKKPRGKMSSYAFFVQTCREEHKKKHPDASVNFSEFSKKCS...,MAASIVRRGMLLARQVVLPQLSPAGKRYLLSSAYVDSHKWEAREKE...,0
176,O94763,Q96M27,MEAPTVETPPDPSPPSAPAPALVPLRAPDVARLREEQEKVVTNCQE...,MMEESGIETTPPGTPPPNPAGLAATAMSSTPVPLAATSSFSSPNVS...,0
177,Q86TM3,Q96M27,MSHWAPEWKRAEANPRDLGASWDVRGSRGSGWSGPFGHQGPRAAGS...,MMEESGIETTPPGTPPPNPAGLAATAMSSTPVPLAATSSFSSPNVS...,0
178,O15151,Q9BY32,MTSFSTSAQCSTSDSACRISPGQINQVRPKLPLLKILHAAGAQGEM...,MAASLVGKKIVFVTGNAKKLEEVVQILGDKFPCTLVAQKIDLPEYQ...,0


In [4]:
from torch.utils.data import TensorDataset

def get_embeddings_df(sequences, model_name):
    model_name_for_file = model_name.replace('/', '-').replace(' ', '-').replace('_', '-')
    f = Path(f'embeddings_{model_name_for_file}.parquet')
    if f.exists():
        return pd.read_parquet(f)

    model = dgeb.get_model(model_name, layers="last", batch_size=1, max_seq_length=2048)

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        embeddings = model.encode(sequences)

    df = pd.DataFrame({'sequence': sequences, 'embedding': list(embeddings.squeeze())})
    df.to_parquet(f)
    return df

sequences = list(set(df_lmdb.seq_1) | set(df_lmdb.seq_2))

embedding_by_sequence = {}
for model_name in MODELS:
    if model_name == 'mint':
        continue
    embeddings_df = get_embeddings_df(sequences, model_name)
    embedding_by_sequence[model_name] = {}
    for elem in embeddings_df.itertuples():
        embedding_by_sequence[model_name][elem.sequence] = elem.embedding


datasets = {}
for model_name in MODELS:
    datasets[model_name] = {}

    if model_name == 'mint':
        for data_type in ['train', 'valid', 'test']:
            env = lmdb.open(f'HumanPPI/normal/{data_type}/')
            operator = env.begin()
            length = int(operator.get("length".encode()))
            labels = []
            with env.begin() as txn:
                for i in range(length):
                    labels.append(
                        json.loads(operator.get(f"{i}".encode()))['label']
                    )
            datasets[model_name][data_type] = TensorDataset(
                torch.load(f"downstream/GeneralPPI/embeddings/HumanPPI/mint_sep/{data_type}.pt"),
                torch.tensor(labels)
            )
        continue

    for data_type in ['train', 'valid', 'test']:
        embeddings = []
        for elem in dfs_lmdb[data_type].itertuples():
            emb1 = embedding_by_sequence[model_name][elem.seq_1]
            emb2 = embedding_by_sequence[model_name][elem.seq_2]
            emb = torch.cat([torch.from_numpy(emb1.copy()), torch.from_numpy(emb2.copy())])
            embeddings.append(emb)

        datasets[model_name][data_type] = TensorDataset(
            torch.stack(embeddings).float(),
            torch.tensor(dfs_lmdb[data_type].label)
        )

In [5]:
import wandb
wandb.login(key="070509e9d7724a43139b65558814901571385078")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ubuntu/.netrc
wandb: Currently logged in as: tehadawest (tehadawest-no) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
from lightning.pytorch import seed_everything
seed_everything(42)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


42

In [7]:
import optuna

import lightning as L

import torch.nn as nn
from sklearn.metrics import accuracy_score
from datetime import datetime


class SimpleMLP(L.LightningModule):
    def __init__(self, input_size, output_size, dropout, lr):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(
            nn.Linear(input_size, input_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(input_size, output_size),
        )
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, x):
        return self.model(x).squeeze()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5)

        # print("optimizer", f"{optimizer}, {optimizer.state_dict()}")
        # print("scheduler", f"{scheduler}, {scheduler.state_dict()}")

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss"
            }
        }

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y.float())
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y.float())
        preds = torch.sigmoid(logits) >= 0.5
        acc = accuracy_score(y.cpu(), preds.cpu())
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_accuracy", acc, prog_bar=True)
        return {"val_loss": loss, "val_accuracy": acc}

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y.float())
        preds = torch.sigmoid(logits) >= 0.5
        acc = accuracy_score(y.cpu(), preds.cpu())
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_accuracy", acc, prog_bar=True)
        return {"test_loss": loss, "test_accuracy": acc}

    def on_train_epoch_end(self):
        lr = self.trainer.optimizers[0].param_groups[0]['lr']
        self.log('lr', lr, prog_bar=True)

In [15]:
best_lightning_models_file = Path('best_lightning_models.csv')
best_optuna_models_file = Path('best_optuna_models.csv')

In [9]:
import logging

logging.getLogger("lightning").setLevel(logging.WARNING)
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

logging.getLogger("lightning.pytorch.trainer.connectors.data_connector").setLevel(logging.ERROR)

# logging.getLogger("optuna").setLevel(logging.WARNING)

In [12]:
from uuid import uuid4

from lightning.pytorch.loggers import WandbLogger, CSVLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from lightning.pytorch.callbacks import Callback


best_lightning_models = []
best_optuna_models = []


def train_my_model(model_name, run_uuid, batch_size, lr, logger, callbacks, enable_progress_bar=False):
    NUM_EPOCHS = 100

    input_size = datasets[model_name]['train'][0][0].shape[-1]
    output_size = 1
    dropout = 0.2

    model = SimpleMLP(input_size, output_size, dropout, lr)

    train_loader = torch.utils.data.DataLoader(
        datasets[model_name]['train'],
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
    )
    val_loader = torch.utils.data.DataLoader(
        datasets[model_name]['valid'],
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
    )

    early_stop = EarlyStopping(monitor="val_accuracy", patience=10, mode="max")

    model_name_for_file = get_model_name_for_file(model_name)
    checkpoint_filename = f"best-model_{model_name_for_file}_{run_uuid}"

    checkpoint = ModelCheckpoint(
        filename=checkpoint_filename,
        monitor="val_accuracy",
        save_top_k=1,
        mode="max",
    )

    trainer = L.Trainer(
        max_epochs=NUM_EPOCHS,
        logger=logger,
        callbacks=[early_stop, checkpoint] + callbacks,
        enable_progress_bar=enable_progress_bar,
    )

    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

    model_info = {
        "model_name": model_name,
        "run_uuid": run_uuid,
        "dropout": dropout,
        "batch_size": batch_size,
        "lr": lr,
        "num_epochs": NUM_EPOCHS,
        "checkpoint_path": checkpoint.best_model_path,
        "train_loss": trainer.callback_metrics["train_loss"].item(),
        "val_accuracy": trainer.callback_metrics["val_accuracy"].item(),
        "epoch": trainer.current_epoch,
        "input_size": input_size,
        "output_size": output_size,
        "datetime": str(datetime.now()),
    }

    return model_info


def get_model_name_for_file(model_name):
    return model_name.replace('/', '-').replace(' ', '-').replace('_', '-')

In [35]:
sum(dfs_lmdb['test'].label)

77

In [ ]:
# class OptunaPruningCallback(Callback):
#     def __init__(self, trial, monitor):
#         self.trial = trial
#         self.monitor = monitor

#     def on_validation_epoch_end(self, trainer, pl_module):
#         current_value = trainer.callback_metrics.get(self.monitor)
#         if current_value is None:
#             return
        
#         self.trial.report(current_value, step=trainer.current_epoch)
#         if self.trial.should_prune():
#             message = f"Trial was pruned at epoch {trainer.current_epoch}."
#             raise optuna.TrialPruned(message)


def create_objective(model_name, run_uuid):
    def objective(trial):
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128, 256, 512])
        lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

        model_name_for_file = get_model_name_for_file(model_name)
        csv_logger = CSVLogger(
            save_dir="optuna_logs",
            name=f"{model_name_for_file}_{run_uuid}",
            version=f"trial-{trial.number}"
        )

        # pruning_callback = OptunaPruningCallback(trial, "val_accuracy")

        # print(f"starting trial: {trial.number} {model_name} {run_uuid} {batch_size} {lr}")
        model_info = train_my_model(model_name, run_uuid, batch_size, lr, csv_logger, [])
        trial.set_user_attr("model_info", model_info)
        best_lightning_models.append(model_info)
        return model_info["val_accuracy"]
    return objective


run_uuid = str(uuid4())

for model_name in MODELS[:]:
    print(f'{model_name=}')
    model_name_for_file = get_model_name_for_file(model_name)

    study = optuna.create_study(
        direction="maximize",
        storage="sqlite:///db.sqlite3",
        study_name=f"my-study_{model_name_for_file}_{run_uuid}",
        # pruner=optuna.pruners.MedianPruner(
        #     n_warmup_steps=10,
        # ),
    )

    objective = create_objective(model_name, run_uuid)
    study.optimize(objective, n_trials=30, n_jobs=4)
    print(f"best trial: val_accuracy={study.best_value:4f} batch_size={study.best_params['batch_size']} lr={study.best_params['lr']} {run_uuid=}")
    best_optuna_models.append(study.best_trial.user_attrs["model_info"])

    # pd.DataFrame(best_lightning_models).to_csv(
    #     best_lightning_models_file,
    #     index=False,
    #     mode='a',
    #     header=not best_lightning_models_file.exists()
    # )

    pd.DataFrame(best_optuna_models).to_csv(
        best_optuna_models_file,
        index=False,
        mode='a',
        header=not best_optuna_models_file.exists()
    )
    best_optuna_models = []
    break

In [ ]:
run_uuid = str(uuid4())

In [13]:
from tqdm.notebook import tqdm

if wandb.run is not None:
    wandb.finish()

best_lightning_models_file = []


print(f'{run_uuid=}')

for model_name in tqdm(MODELS):
    for batch_size in tqdm([16, 32, 64, 128, 256, 512], desc=f'{model_name=}', leave=False):
        for lr in tqdm([1e-5, 1e-4, 1e-3, 1e-2, 1e-1], desc=f'{batch_size=}', leave=False):

            wandb_logger = WandbLogger(
                config={
                    "model_name": model_name,
                    "batch_size": batch_size,
                    "lr": lr,
                },
                settings=wandb.Settings(silent=True)
            )

            model_info = train_my_model(model_name, run_uuid, batch_size, lr, wandb_logger, [])
            model_info['wandb_url'] = wandb_logger.experiment.url
            best_lightning_models.append(model_info)

            wandb.finish()

run_uuid='55e4deb8-97af-41d7-a25f-e61e7d2e7f62'


  0%|          | 0/11 [00:00<?, ?it/s]

model_name='esm3_sm_open_v1':   0%|          | 0/6 [00:00<?, ?it/s]

batch_size=16:   0%|          | 0/5 [00:00<?, ?it/s]

batch_size=32:   0%|          | 0/5 [00:00<?, ?it/s]

...

batch_size=32:   0%|          | 0/5 [00:00<?, ?it/s]

batch_size=64:   0%|          | 0/5 [00:00<?, ?it/s]

batch_size=128:   0%|          | 0/5 [00:00<?, ?it/s]

batch_size=256:   0%|          | 0/5 [00:00<?, ?it/s]

batch_size=512:   0%|          | 0/5 [00:00<?, ?it/s]

In [16]:
pd.DataFrame(best_lightning_models).to_csv(
    best_lightning_models_file,
    index=False,
    mode='a',
    header=not best_lightning_models_file.exists()
)

In [18]:
df = pd.read_csv(best_lightning_models_file)
df.sort_values(by='val_accuracy', ascending=False)[:10]

,model_name,run_uuid,dropout,batch_size,lr,num_epochs,checkpoint_path,train_loss,val_accuracy,epoch,input_size,output_size,datetime,wandb_url
118,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,512,0.01000,100,./lightning_logs/edv6pc68/checkpoints/best-mod...,0.107111,0.905983,15,5120,1,2025-07-18 15:57:46.108395,https://wandb.ai/tehadawest-no/lightning_logs/...
100,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,64,0.00001,100,./lightning_logs/eqltn71m/checkpoints/best-mod...,0.103136,0.901709,20,5120,1,2025-07-18 15:47:40.449079,https://wandb.ai/tehadawest-no/lightning_logs/...
110,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,256,0.00001,100,./lightning_logs/505n3h4o/checkpoints/best-mod...,0.117083,0.901709,29,5120,1,2025-07-18 15:54:04.917737,https://wandb.ai/tehadawest-no/lightning_logs/...
95,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,32,0.00001,100,./lightning_logs/cy6xigcp/checkpoints/best-mod...,0.058701,0.897436,20,5120,1,2025-07-18 15:43:22.963701,https://wandb.ai/tehadawest-no/lightning_logs/...
115,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,512,0.00001,100,./lightning_logs/v7e313qw/checkpoints/best-mod...,0.185232,0.897436,18,5120,1,2025-07-18 15:56:25.828598,https://wandb.ai/tehadawest-no/lightning_logs/...
103,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,64,0.01000,100,./lightning_logs/32jmab8i/checkpoints/best-mod...,0.391689,0.897436,24,5120,1,2025-07-18 15:49:58.872573,https://wandb.ai/tehadawest-no/lightning_logs/...
105,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,128,0.00001,100,./lightning_logs/gul8jgyt/checkpoints/best-mod...,0.074542,0.897436,25,5120,1,2025-07-18 15:51:14.475701,https://wandb.ai/tehadawest-no/lightning_logs/...
90,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,16,0.00001,100,./lightning_logs/v58q88qm/checkpoints/best-mod...,0.291557,0.893162,16,5120,1,2025-07-18 15:37:19.184977,https://wandb.ai/tehadawest-no/lightning_logs/...
109,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,128,0.10000,100,./lightning_logs/26v0ukva/checkpoints/best-mod...,0.179807,0.888889,27,5120,1,2025-07-18 15:53:20.053061,https://wandb.ai/tehadawest-no/lightning_logs/...
260,facebook/esm2_t36_3B_UR50D,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,256,0.00001,100,./lightning_logs/zz8cbx9o/checkpoints/best-mod...,0.107762,0.888889,13,5120,1,2025-07-18 17:26:52.992997,https://wandb.ai/tehadawest-no/lightning_logs/...


In [61]:
df = pd.read_csv(best_models_info_file)
# df.sort_values(by='valid_accuracy', ascending=False)

In [22]:
import numpy as np

def get_accuracy_and_loss(dataloader, model, loss_fn, device):
    model.eval()
    losses = []
    preds = []
    targets = []
    with torch.no_grad():
        for (embs, target) in dataloader:
            embs = embs.to(device)
            target = target.to(device)
            pred = model(embs).squeeze(-1)
            loss = loss_fn(pred.squeeze(), target.float())
            pred = torch.sigmoid(pred)
            preds.append(pred.detach().cpu().numpy())
            targets.append(target.cpu().numpy())
            losses.append(loss.detach().cpu().item())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    threshold = 0.5
    binary_predictions = (preds >= threshold).astype(int)
    accuracy = accuracy_score(targets, binary_predictions)

    valid_loss = np.mean(losses)
    return accuracy, valid_loss


test_accuracies = []
test_losses = []
for elem in df.itertuples():
    test_loader = torch.utils.data.DataLoader(
        datasets[elem.model_name]['test'], batch_size=elem.batch_size, shuffle=False
    )

    model = SimpleMLP.load_from_checkpoint(elem.checkpoint_path)

    loss_fn = nn.BCEWithLogitsLoss()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    test_accuracy, test_loss = get_accuracy_and_loss(test_loader, model, loss_fn, device)

    test_accuracies.append(test_accuracy)
    test_losses.append(test_loss)

df['test_accuracy'] = test_accuracies
df['test_loss'] = test_losses

In [ ]:
# 1. test acc > val acc -- seems fishy... -- could be due to bad split of dataset -- possible fix to resplit manually randomly
# 2.

In [31]:
df.sort_values(by='test_accuracy', ascending=False).reset_index(drop=True)[:20]

,model_name,run_uuid,dropout,batch_size,lr,num_epochs,checkpoint_path,train_loss,val_accuracy,epoch,input_size,output_size,datetime,wandb_url,test_accuracy,test_loss
0,facebook/esm2_t33_650M_UR50D,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,64,0.01000,100,./lightning_logs/9yvkuafh/checkpoints/best-model_facebook-esm2-t33-650M-UR50D_55e4deb8-97af-41d7...,0.054428,0.824786,15,2560,1,2025-07-18 17:06:49.595098,https://wandb.ai/tehadawest-no/lightning_logs/runs/9yvkuafh,0.922222,0.332406
1,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,64,0.01000,100,./lightning_logs/32jmab8i/checkpoints/best-model_esmc-6b-2024-12_55e4deb8-97af-41d7-a25f-e61e7d2...,0.391689,0.897436,24,5120,1,2025-07-18 15:49:58.872573,https://wandb.ai/tehadawest-no/lightning_logs/runs/32jmab8i,0.916667,0.362017
2,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,128,0.00001,100,./lightning_logs/gul8jgyt/checkpoints/best-model_esmc-6b-2024-12_55e4deb8-97af-41d7-a25f-e61e7d2...,0.074542,0.897436,25,5120,1,2025-07-18 15:51:14.475701,https://wandb.ai/tehadawest-no/lightning_logs/runs/gul8jgyt,0.911111,0.251861
3,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,512,0.10000,100,./lightning_logs/pcfcb436/checkpoints/best-model_esmc-6b-2024-12_55e4deb8-97af-41d7-a25f-e61e7d2...,0.146217,0.884615,24,5120,1,2025-07-18 15:58:22.478569,https://wandb.ai/tehadawest-no/lightning_logs/runs/pcfcb436,0.905556,0.285363
4,esm3-large-2024-03,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,128,0.01000,100,./lightning_logs/g51dwik6/checkpoints/best-model_esm3-large-2024-03_55e4deb8-97af-41d7-a25f-e61e...,0.083971,0.867521,22,12288,1,2025-07-18 15:25:07.955953,https://wandb.ai/tehadawest-no/lightning_logs/runs/g51dwik6,0.905556,0.331771
5,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,512,0.00010,100,./lightning_logs/xae28ksd/checkpoints/best-model_esmc-6b-2024-12_55e4deb8-97af-41d7-a25f-e61e7d2...,0.044519,0.858974,21,5120,1,2025-07-18 15:56:56.401475,https://wandb.ai/tehadawest-no/lightning_logs/runs/xae28ksd,0.905556,0.285962
6,facebook/esm2_t36_3B_UR50D,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,32,0.00010,100,./lightning_logs/z4qvkoin/checkpoints/best-model_facebook-esm2-t36-3B-UR50D_55e4deb8-97af-41d7-a...,0.000947,0.841880,12,5120,1,2025-07-18 17:19:36.596124,https://wandb.ai/tehadawest-no/lightning_logs/runs/z4qvkoin,0.905556,0.303300
7,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,512,0.00100,100,./lightning_logs/6m2yt3a6/checkpoints/best-model_esmc-6b-2024-12_55e4deb8-97af-41d7-a25f-e61e7d2...,0.060383,0.863248,18,5120,1,2025-07-18 15:57:23.159414,https://wandb.ai/tehadawest-no/lightning_logs/runs/6m2yt3a6,0.900000,0.295144
8,esmc-6b-2024-12,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,256,0.01000,100,./lightning_logs/1gemckh1/checkpoints/best-model_esmc-6b-2024-12_55e4deb8-97af-41d7-a25f-e61e7d2...,0.109050,0.863248,15,5120,1,2025-07-18 15:55:32.862604,https://wandb.ai/tehadawest-no/lightning_logs/runs/1gemckh1,0.900000,0.300534
9,esm3-large-2024-03,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,32,0.00001,100,./lightning_logs/5fu4q9pw/checkpoints/best-model_esm3-large-2024-03_55e4deb8-97af-41d7-a25f-e61e...,0.083867,0.863248,15,12288,1,2025-07-18 15:07:06.815874,https://wandb.ai/tehadawest-no/lightning_logs/runs/5fu4q9pw,0.900000,0.262640


In [30]:
df.sort_values(by='test_accuracy', ascending=False).reset_index(drop=True)[lambda df: df.model_name == 'mint']

,model_name,run_uuid,dropout,batch_size,lr,num_epochs,checkpoint_path,train_loss,val_accuracy,epoch,input_size,output_size,datetime,wandb_url,test_accuracy,test_loss
63,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,32,0.00100,100,./lightning_logs/hiird4jk/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.040198,0.829060,13,2560,1,2025-07-18 18:08:57.859639,https://wandb.ai/tehadawest-no/lightning_logs/runs/hiird4jk,0.877778,0.388219
72,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,16,0.01000,100,./lightning_logs/jmbjik29/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.040891,0.846154,22,2560,1,2025-07-18 18:05:17.133175,https://wandb.ai/tehadawest-no/lightning_logs/runs/jmbjik29,0.877778,0.551595
78,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,16,0.00100,100,./lightning_logs/wfow3hjl/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.016771,0.829060,11,2560,1,2025-07-18 18:03:25.498709,https://wandb.ai/tehadawest-no/lightning_logs/runs/wfow3hjl,0.872222,0.414887
79,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,512,0.01000,100,./lightning_logs/id8sa9t3/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.064767,0.841880,15,2560,1,2025-07-18 18:20:52.214892,https://wandb.ai/tehadawest-no/lightning_logs/runs/id8sa9t3,0.872222,0.368643
87,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,64,0.01000,100,./lightning_logs/pv0ujm2t/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.141430,0.837607,16,2560,1,2025-07-18 18:13:34.690844,https://wandb.ai/tehadawest-no/lightning_logs/runs/pv0ujm2t,0.872222,0.433210
92,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,128,0.00010,100,./lightning_logs/hxrmvqyh/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.248026,0.858974,13,2560,1,2025-07-18 18:15:14.673068,https://wandb.ai/tehadawest-no/lightning_logs/runs/hxrmvqyh,0.866667,0.318109
104,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,16,0.00010,100,./lightning_logs/o34fnmei/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.007545,0.833333,13,2560,1,2025-07-18 18:02:28.530251,https://wandb.ai/tehadawest-no/lightning_logs/runs/o34fnmei,0.866667,0.391032
107,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,64,0.00010,100,./lightning_logs/ogy1wngv/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.021502,0.850427,12,2560,1,2025-07-18 18:12:25.857792,https://wandb.ai/tehadawest-no/lightning_logs/runs/ogy1wngv,0.866667,0.338184
112,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,32,0.10000,100,./lightning_logs/1q0u63s7/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.231902,0.858974,20,2560,1,2025-07-18 18:11:10.960579,https://wandb.ai/tehadawest-no/lightning_logs/runs/1q0u63s7,0.866667,0.412277
129,mint,55e4deb8-97af-41d7-a25f-e61e7d2e7f62,0.2,32,0.00010,100,./lightning_logs/gscpz0j5/checkpoints/best-model_mint_55e4deb8-97af-41d7-a25f-e61e7d2e7f62.ckpt,0.111456,0.854701,22,2560,1,2025-07-18 18:08:15.708048,https://wandb.ai/tehadawest-no/lightning_logs/runs/gscpz0j5,0.861111,0.392872


In [29]:
pd.set_option('display.max_colwidth', 100)

In [59]:
loss_fn = nn.BCEWithLogitsLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
get_accuracy_and_loss(test_loader, model, loss_fn, device)

(0.8222222222222222, np.float64(0.3580707311630249))

In [39]:
def do_stuff():
    model_name = "mint"

    input_size = datasets[model_name]['train'][0][0].shape[-1]
    output_size = 1

    batch_size = 16
    lr = 1e-4

    dropout = 0.2

    NUM_EPOCHS = 100

    model = SimpleMLP(input_size, output_size, dropout, lr)

    train_loader = torch.utils.data.DataLoader(
        datasets[model_name]['train'], batch_size=batch_size, shuffle=True, num_workers=3
    )
    val_loader = torch.utils.data.DataLoader(
        datasets[model_name]['valid'], batch_size=batch_size, shuffle=False, num_workers=3
    )

    from lightning.pytorch.loggers import WandbLogger

    if wandb.run is not None:
        wandb.finish()
    wandb_logger = WandbLogger(
        project="mint",
        config={
            "model_name": model_name,
            "batch_size": batch_size,
            "num_epochs": NUM_EPOCHS,
        },
    )

    trainer = L.Trainer(
        max_epochs=NUM_EPOCHS,
        logger=wandb_logger,
    )

    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

    model_info = {
        "model_name": model_name,
        "dropout": dropout,
        "batch_size": batch_size,
        "lr": lr,
        "num_epochs": NUM_EPOCHS,
        "train_loss": trainer.callback_metrics["train_loss"],
        "valid_accuracy": trainer.callback_metrics["val_accuracy"],
        "epoch": trainer.current_epoch,
        "wandb_url": wandb_logger.experiment.url,
        "input_size": input_size,
        "output_size": output_size,
        "datetime": str(datetime.now()),
    }

    print(model_info)

    test_loader = torch.utils.data.DataLoader(
        datasets[model_name]['test'], batch_size=batch_size, shuffle=False, num_workers=3,
    )

    trainer.test(model, dataloaders=test_loader)
    wandb.finish()

do_stuff()

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name    | Type              | Params | Mode 
------------------------------------------------------
0 | model   | Sequential        | 6.6 M  | train
1 | loss_fn | BCEWithLogitsLoss | 0      | train
------------------------------------------------------
6.6 M     Trainable params
0         Non-trainable params
6.6 M     Total params
26.235    Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode
INFO:lightning.pytorch.callbacks.model_summary:
  | Name    | Type              | Params | Mode 
------------------------------------------------------
0 | model   | Sequential        | 6.6 M  | train
1 | loss_fn | BCEWithLogitsLoss | 0      | train
------------------------------------------------------
6.6 M     Trainable params
0         Non-trainable params
6.6 M     Total params
26.235    Total estimated 

optimizer AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.01
), {'state': {}, 'param_groups': [{'lr': 0.0001, 'betas': (0.9, 0.999), 'eps': 1e-08, 'weight_decay': 0.01, 'amsgrad': False, 'maximize': False, 'foreach': None, 'capturable': False, 'differentiable': False, 'fused': None, 'decoupled_weight_decay': True, 'params': [0, 1, 2, 3]}]}
scheduler <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x762cb2cbeb60>, {'factor': 0.1, 'default_min_lr': 0, 'min_lrs': [0], 'patience': 5, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'eps': 1e-08, 'last_epoch': 0, '_last_lr': [0.0001], 'mode_worse': inf, 'best': inf, 'num_bad_epochs': 0}


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


{'model_name': 'mint', 'dropout': 0.2, 'batch_size': 16, 'lr': 0.0001, 'num_epochs': 100, 'train_loss': tensor(0.5437), 'valid_accuracy': tensor(0.8291), 'epoch': 100, 'wandb_url': 'https://wandb.ai/tehadawest-no/mint/runs/tii5ku5t', 'input_size': 2560, 'output_size': 1, 'datetime': '2025-07-18 06:39:07.317912'}


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.8666666746139526     │
│         test_loss         │    0.4401863217353821     │
└───────────────────────────┴───────────────────────────┘

epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇█
lr,███▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
test_loss,▁
train_loss,██▆▃▃▄▂▁▁▁▁▆▃▁▂▂▁▂▁▃▄▃▂▁▃▄▂▃▂▄▂▂▅▃▁▂▂▄▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█
val_accuracy,▁▇█▅▅▅▄▄▄▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅
val_loss,▁▆▇▇████████████████████████████████████
epoch,100
lr,0.0
test_accuracy,0.86667


In [34]:
from torch import nn
import numpy as np
from sklearn.metrics import accuracy_score


torch.manual_seed(0)


class SimpleMLP(nn.Module):
    def __init__(self, input_size, output_size, dropout):
        super().__init__()

        self.project = nn.Sequential(
            nn.Linear(input_size, input_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(input_size, output_size),
        )

    def forward(self, x):
        return self.project(x)


def train(dataloader, model, loss_fn, optimizer, device):
    # losses = []
    loss_accum = 0
    for step, (embs, target) in enumerate(dataloader):
        model.train()
        optimizer.zero_grad()
        embs = embs.to(device)
        target = target.to(device)
        pred = model(embs)
        loss = loss_fn(pred.squeeze(), target.float())
        loss.backward()
        optimizer.step()
        # losses.append(loss.detach().cpu().item())
        loss_accum += loss.detach().cpu().item()
    # train_loss = np.mean(losses)
    train_loss = loss_accum / (step + 1)
    return train_loss


def get_accuracy_and_loss(dataloader, model, loss_fn, device):
    # model.eval()
    losses = []
    preds = []
    targets = []
    with torch.no_grad():
        for (embs, target) in dataloader:
            embs = embs.to(device)
            target = target.to(device)
            pred = model(embs).squeeze(-1)
            loss = loss_fn(pred.squeeze(), target.float())
            pred = torch.sigmoid(pred)
            preds.append(pred.detach().cpu().numpy())
            targets.append(target.cpu().numpy())
            losses.append(loss.detach().cpu().item())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    threshold = 0.5
    binary_predictions = (preds >= threshold).astype(int)
    accuracy = accuracy_score(targets, binary_predictions)

    valid_loss = np.mean(losses)
    return accuracy, valid_loss



model_name = "mint"


batch_size = 16
lr = 1e-4
dropout = 0.2

num_epochs = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_size = datasets[model_name]['train'][0][0].shape[-1]
output_size = 1


train_loader = torch.utils.data.DataLoader(
    datasets[model_name]['train'], batch_size=batch_size, shuffle=True
)
valid_loader = torch.utils.data.DataLoader(
    datasets[model_name]['valid'], batch_size=batch_size, shuffle=False
)


model = SimpleMLP(input_size, output_size, dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min"
)
loss_fn = nn.BCEWithLogitsLoss()

best_valid_accuracy = 0

for epoch in range(num_epochs):
    train_loss = train(train_loader, model, loss_fn, optimizer, device)
    valid_accuracy, valid_loss = get_accuracy_and_loss(valid_loader, model, loss_fn, device)
    scheduler.step(train_loss)

    if valid_accuracy > best_valid_accuracy:
        best_valid_accuracy = valid_accuracy

    print({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_accuracy,
    })

test_loader = torch.utils.data.DataLoader(
    datasets[model_name]['test'], batch_size=batch_size, shuffle=False
)
test_accuracy, test_loss = get_accuracy_and_loss(test_loader, model, loss_fn, device)
print({
    "test_accuracy": test_accuracy,
    "test_loss": test_loss,
})

{'epoch': 0, 'train_loss': 0.33512011189183566, 'valid_loss': np.float64(0.37977131009101867), 'valid_accuracy': 0.8632478632478633}
{'epoch': 1, 'train_loss': 0.23151742964643293, 'valid_loss': np.float64(0.41113057881593706), 'valid_accuracy': 0.8461538461538461}


KeyboardInterrupt: 